# MediKiosk dataset audit and exploratory analysis

This notebook runs the repository's reusable EDA module and displays aggregate-only outputs. It does not display patient names, IDs, source text, or row-level examples. Counts are calculated from files currently on disk; external manifests are not treated as data.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

repo = Path.cwd()
if not (repo / "backend" / "training" / "eda.py").is_file():
    raise RuntimeError("Open this notebook from the repository root.")
env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "backend")
subprocess.run([sys.executable, "-m", "training.eda"], cwd=repo / "backend", env=env, check=True)
eda_dir = repo / "evaluation_results" / "eda"
report = json.loads((eda_dir / "dataset_eda.json").read_text(encoding="utf-8"))
report["_summary"]

In [ ]:
files = [(name, item["rows"], item["columns"], item["missing_cell_share"], item["exact_duplicate_rows"])
         for name, item in report.items() if not name.startswith("_")]
print("file | rows | columns | missing-cell share | exact duplicate rows")
for row in files:
    print(f"{row[0]} | {row[1]} | {row[2]} | {row[3]:.2%} | {row[4]}")

In [ ]:
from IPython.display import display, Image

for filename in ("dataset_composition.png", "missingness.png", "text_lengths.png", "text_length_distributions.png", "class_frequencies.png"):
    path = eda_dir / filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path)))

## Interpretation and task suitability

- The history interview data support a patient-grouped, synthetic next-question imitation experiment.
- Lab and prescription tables contain field-level values but no source-text character offsets, so they do not supervise token-classification NER.
- OCR pairs are simulated text-to-text examples; image files are not bundled.
- The 300 history-summary examples are templated synthetic data. ACI-Bench dialogue/note data are paired but licensing and source-evidence alignment are unresolved. MTSamples is note-only.
- ASR rows contain simulated transcript text; there are no bundled audio files.

Review `docs/DATASET_AUDIT.md` and `docs/DATASET_REGISTER.md` for provenance and limitations before using any source.